# 30 · Gold: Date dimension
**Owner: Ana** · Layer: gold · Runs after: nothing (generated, not loaded from a file)

**What:** one row per calendar day from 1 Jan 2011 to 31 Dec 2026, plus an Unknown row (key −1).

**Why it is generated, not imported:** a date dimension is pure calendar logic, so building it in code guarantees there are no gaps.

**Design choices to be ready to explain in the defence**
- **Key `yyyymmdd`** (for example `20260304`), not an auto-number: it is readable, sorts in date order, and any date can be turned into its key without a lookup.
- **Australian financial year** runs July to June and is named by the year it ends: 4 Mar 2026 is FY2026. ICA labels the same event "FY25" (semantic inconsistency SEM1), so we derive our own.
- **Southern-hemisphere seasons:** summer is December to February.
- It is a **role-playing dimension**: the event fact uses it twice, for the start date and the end date.

This notebook also drops the gold fact tables first, because a dimension can't be replaced while a fact's foreign key points to it.

In [0]:
%run ./00_common

In [0]:
for t in ["fact_event_loss", "fact_state_year"]:
    spark.sql(f"DROP TABLE IF EXISTS {tname('gold', t)}")

days = spark.sql(f"SELECT explode(sequence(DATE'{SCOPE_START}', DATE'{SCOPE_END_YEAR}-12-31', INTERVAL 1 DAY)) AS full_date")
dim_date = days.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    "full_date",
    F.dayofmonth("full_date").alias("day_of_month"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.month("full_date").alias("month_number"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.quarter("full_date").alias("calendar_quarter"),
    F.year("full_date").alias("calendar_year"),
    F.when(F.month("full_date") >= 7, F.year("full_date") + 1).otherwise(F.year("full_date")).alias("au_financial_year"),
    F.when(F.month("full_date").isin(12, 1, 2), "Summer").when(F.month("full_date").isin(3, 4, 5), "Autumn")
     .when(F.month("full_date").isin(6, 7, 8), "Winter").otherwise("Spring").alias("season"),
    F.dayofweek("full_date").isin(1, 7).alias("is_weekend"),
)
n = write(with_unknown(dim_date, date_key=-1, day_name="Unknown", month_name="Unknown", season="Unknown"), "gold", "dim_date")
expected = spark.sql(f"SELECT datediff(DATE'{SCOPE_END_YEAR}-12-31', DATE'{SCOPE_START}') + 1").first()[0] + 1
log_step("gold", "build_dim_date", "gold.dim_date", expected, n, 0, "generated calendar + Unknown row")

In [0]:
display(read("gold", "dim_date").where("date_key IN (20110101, 20260304, 20260630, -1)"))